In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import warnings
import logging
import os
import time
from joblib import Parallel, delayed
from prophet import Prophet
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").setLevel(logging.ERROR)
logging.getLogger("prophet").setLevel(logging.ERROR)


In [3]:
import os
df = pd.read_csv('data/clt_pipe_HD_sales_0_95_transformed.csv')
df = df.drop(columns = ['Unnamed: 5'])
df = df.rename(columns={'sku nbr': 'sku_nbr', 'sub class': 'sub_class', 'sales units': 'sales_units', 'sku name': 'sku_name'})
# df.head()

In [4]:
df = df.copy()
unique_sku_nbr = df['sku_nbr'].unique().tolist()
unique_sku_nbr[1]

144745

In [5]:
forecast_horizon = 12        # final future forecast length
validation_horizon = 3       # each rolling validation window
initial_train_size = 24      # first rolling fold trains on first 24 months
step_size = 3                # move forward 3 months per fold

min_months_required = 24
long_history_threshold = 36
max_skus = min(166, len(unique_sku_nbr))

min_months_for_rolling_validation = initial_train_size + validation_horizon

# Parallel settings
n_jobs = max(1, os.cpu_count() - 1)

# Recommended False for parallel processing.
# Returning full Prophet / SARIMA model objects from worker processes can use a lot of memory.
SAVE_MODEL_OBJECTS = False

df_model = df.copy()
df_model["month"] = pd.to_datetime(df_model["month"])
df_model["month"] = df_model["month"].dt.to_period("M").dt.to_timestamp()

def calculate_mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mask = y_true != 0

    if mask.sum() == 0:
        return np.inf

    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def clean_forecast_values(values):
    values = np.array(values)
    values = np.where(values < 0, 0, values)
    return values

def get_future_months(last_month, forecast_horizon):
    return pd.date_range(
        start=last_month + pd.DateOffset(months=1),
        periods=forecast_horizon,
        freq="MS"
    )

def make_rolling_folds(temp_ts, initial_train_size, validation_horizon, step_size):
    folds = []
    n = len(temp_ts)
    train_end = initial_train_size

    while train_end + validation_horizon <= n:
        train_fold = temp_ts.iloc[:train_end].copy()
        validation_fold = temp_ts.iloc[
            train_end:train_end + validation_horizon
        ].copy()

        folds.append((train_fold, validation_fold))
        train_end += step_size

    return folds

def fit_predict_prophet(train_df, horizon, params):
    prophet_train = train_df[["month", "sales_units"]].rename(
        columns={"month": "ds", "sales_units": "y"}
    )

    model = Prophet(
        changepoint_prior_scale=params["changepoint_prior_scale"],
        seasonality_prior_scale=params["seasonality_prior_scale"],
        seasonality_mode=params["seasonality_mode"],
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False
    )

    model.fit(prophet_train)

    future = model.make_future_dataframe(
        periods=horizon,
        freq="MS"
    )

    forecast = model.predict(future)

    forecast_values = forecast["yhat"].tail(horizon).values
    forecast_values = clean_forecast_values(forecast_values)

    return model, forecast_values


def fit_predict_sarima(train_df, horizon, params):
    train_y = train_df.set_index("month")["sales_units"]
    train_y = train_y.asfreq("MS")

    model = SARIMAX(
        train_y,
        order=params["order"],
        seasonal_order=params["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    fitted_model = model.fit(disp=False)

    forecast_values = fitted_model.forecast(steps=horizon).values
    forecast_values = clean_forecast_values(forecast_values)

    return fitted_model, forecast_values


def fit_predict_ses(train_df, horizon):
    train_y = train_df.set_index("month")["sales_units"]
    train_y = train_y.asfreq("MS")

    model = SimpleExpSmoothing(
        train_y,
        initialization_method="estimated"
    )

    fitted_model = model.fit(optimized=True)

    forecast_values = fitted_model.forecast(horizon).values
    forecast_values = clean_forecast_values(forecast_values)

    return fitted_model, forecast_values


def fit_predict_des(train_df, horizon):
    train_y = train_df.set_index("month")["sales_units"]
    train_y = train_y.asfreq("MS")

    model = ExponentialSmoothing(
        train_y,
        trend="add",
        seasonal=None,
        initialization_method="estimated"
    )

    fitted_model = model.fit(optimized=True)

    forecast_values = fitted_model.forecast(horizon).values
    forecast_values = clean_forecast_values(forecast_values)

    return fitted_model, forecast_values


def fit_predict_tes(train_df, horizon):
    train_y = train_df.set_index("month")["sales_units"]
    train_y = train_y.asfreq("MS")

    model = ExponentialSmoothing(
        train_y,
        trend="add",
        seasonal="add",
        seasonal_periods=12,
        initialization_method="estimated"
    )

    fitted_model = model.fit(optimized=True)

    forecast_values = fitted_model.forecast(horizon).values
    forecast_values = clean_forecast_values(forecast_values)

    return fitted_model, forecast_values

def rolling_validate_model(
    temp_ts,
    model_type,
    params=None,
    initial_train_size=24,
    validation_horizon=3,
    step_size=3
):
    folds = make_rolling_folds(
        temp_ts=temp_ts,
        initial_train_size=initial_train_size,
        validation_horizon=validation_horizon,
        step_size=step_size
    )

    fold_mapes = []

    for fold_number, (train_fold, validation_fold) in enumerate(folds, start=1):

        y_validation = validation_fold["sales_units"].values

        try:
            if model_type == "prophet":
                _, forecast_values = fit_predict_prophet(
                    train_fold,
                    validation_horizon,
                    params
                )

            elif model_type == "SARIMA":
                _, forecast_values = fit_predict_sarima(
                    train_fold,
                    validation_horizon,
                    params
                )

            elif model_type == "SES":
                _, forecast_values = fit_predict_ses(
                    train_fold,
                    validation_horizon
                )

            elif model_type == "DES":
                _, forecast_values = fit_predict_des(
                    train_fold,
                    validation_horizon
                )

            elif model_type == "TES":
                if len(train_fold) < 24:
                    continue

                _, forecast_values = fit_predict_tes(
                    train_fold,
                    validation_horizon
                )

            else:
                continue

            mape = calculate_mape(y_validation, forecast_values)

            if np.isfinite(mape):
                fold_mapes.append(mape)

        except Exception:
            continue

    if len(fold_mapes) == 0:
        return None

    return {
        "model_type": model_type,
        "params": params,
        "avg_mape": np.mean(fold_mapes),
        "fold_mapes": fold_mapes,
        "n_successful_folds": len(fold_mapes)
    }

prophet_param_grid = [
    {
        "changepoint_prior_scale": cps,
        "seasonality_prior_scale": sps,
        "seasonality_mode": sm
    }
    for cps in [0.01, 0.05, 0.10]
    for sps in [1.0, 5.0, 10.0]
    for sm in ["additive", "multiplicative"]
]

# Reduced SARIMA grid: 8 combinations
sarima_param_grid = [
    {
        "order": (p, d, q),
        "seasonal_order": (P, D, Q, 12)
    }
    for p in [0, 1]
    for d in [1]
    for q in [0, 1]
    for P in [0]
    for D in [1]
    for Q in [0, 1]
]

print(f"SARIMA combinations: {len(sarima_param_grid)}")
print(f"CPU cores available: {os.cpu_count()}")
print(f"Parallel jobs to use: {n_jobs}")

SARIMA combinations: 8
CPU cores available: 16
Parallel jobs to use: 15


In [6]:
def process_one_sku(current_sku, save_model_objects=False):

    try:
        temp_df = df_model[df_model["sku_nbr"] == current_sku].copy()
        temp_df = temp_df.sort_values("month")

        if temp_df.empty:
            return {
                "sku_nbr": current_sku,
                "forecast": None,
                "metrics": {
                    "sku_nbr": current_sku,
                    "sku_name": None,
                    "sub_class": None,
                    "status": "skipped",
                    "reason": "no rows found for SKU",
                    "n_months": 0
                },
                "model_entry": None
            }

        sku_name = temp_df["sku_name"].iloc[0]
        sub_class = temp_df["sub_class"].iloc[0]

        # Aggregate to one row per month per SKU
        temp_ts = (
            temp_df
            .groupby("month", as_index=False)
            .agg({"sales_units": "sum"})
            .sort_values("month")
        )

        # Create continuous monthly time series
        full_month_range = pd.date_range(
            start=temp_ts["month"].min(),
            end=temp_ts["month"].max(),
            freq="MS"
        )

        temp_ts = (
            temp_ts
            .set_index("month")
            .reindex(full_month_range)
            .rename_axis("month")
            .reset_index()
        )

        # Missing months treated as zero sales
        temp_ts["sales_units"] = temp_ts["sales_units"].fillna(0)

        temp_ts["sku_nbr"] = current_sku
        temp_ts["sku_name"] = sku_name
        temp_ts["sub_class"] = sub_class

        temp_ts = temp_ts[
            ["sku_nbr", "sub_class", "sales_units", "month", "sku_name"]
        ]

        n_months = temp_ts["month"].nunique()

        if n_months < min_months_required:
            return {
                "sku_nbr": current_sku,
                "forecast": None,
                "metrics": {
                    "sku_nbr": current_sku,
                    "sku_name": sku_name,
                    "sub_class": sub_class,
                    "status": "skipped",
                    "reason": "not enough history",
                    "n_months": n_months
                },
                "model_entry": None
            }

        folds = make_rolling_folds(
            temp_ts=temp_ts,
            initial_train_size=initial_train_size,
            validation_horizon=validation_horizon,
            step_size=step_size
        )

        if len(folds) == 0:
            return {
                "sku_nbr": current_sku,
                "forecast": None,
                "metrics": {
                    "sku_nbr": current_sku,
                    "sku_name": sku_name,
                    "sub_class": sub_class,
                    "status": "skipped",
                    "reason": "not enough data for rolling validation",
                    "n_months": n_months,
                    "minimum_needed_for_rolling_validation": min_months_for_rolling_validation
                },
                "model_entry": None
            }

        candidate_results = []

        if n_months >= long_history_threshold:

            for params in prophet_param_grid:
                result = rolling_validate_model(
                    temp_ts=temp_ts,
                    model_type="prophet",
                    params=params,
                    initial_train_size=initial_train_size,
                    validation_horizon=validation_horizon,
                    step_size=step_size
                )

                if result is not None:
                    candidate_results.append(result)

            for params in sarima_param_grid:
                result = rolling_validate_model(
                    temp_ts=temp_ts,
                    model_type="SARIMA",
                    params=params,
                    initial_train_size=initial_train_size,
                    validation_horizon=validation_horizon,
                    step_size=step_size
                )

                if result is not None:
                    candidate_results.append(result)

        else:

            for model_type in ["SES", "DES", "TES"]:
                result = rolling_validate_model(
                    temp_ts=temp_ts,
                    model_type=model_type,
                    params=None,
                    initial_train_size=initial_train_size,
                    validation_horizon=validation_horizon,
                    step_size=step_size
                )

                if result is not None:
                    candidate_results.append(result)

        if len(candidate_results) == 0:
            return {
                "sku_nbr": current_sku,
                "forecast": None,
                "metrics": {
                    "sku_nbr": current_sku,
                    "sku_name": sku_name,
                    "sub_class": sub_class,
                    "status": "skipped",
                    "reason": "no model successfully completed rolling validation",
                    "n_months": n_months
                },
                "model_entry": None
            }

        # Best model by lowest average rolling MAPE
        best_result = min(candidate_results, key=lambda x: x["avg_mape"])

        best_model_type = best_result["model_type"]
        best_params = best_result["params"]
        best_avg_mape = best_result["avg_mape"]
        best_fold_mapes = best_result["fold_mapes"]
        best_n_successful_folds = best_result["n_successful_folds"]

        try:
            if best_model_type == "prophet":
                final_model, final_forecast_values = fit_predict_prophet(
                    temp_ts,
                    forecast_horizon,
                    best_params
                )

            elif best_model_type == "SARIMA":
                final_model, final_forecast_values = fit_predict_sarima(
                    temp_ts,
                    forecast_horizon,
                    best_params
                )

            elif best_model_type == "SES":
                final_model, final_forecast_values = fit_predict_ses(
                    temp_ts,
                    forecast_horizon
                )

            elif best_model_type == "DES":
                final_model, final_forecast_values = fit_predict_des(
                    temp_ts,
                    forecast_horizon
                )

            elif best_model_type == "TES":
                final_model, final_forecast_values = fit_predict_tes(
                    temp_ts,
                    forecast_horizon
                )

            else:
                raise ValueError(f"Unknown best model type: {best_model_type}")

        except Exception as e:
            return {
                "sku_nbr": current_sku,
                "forecast": None,
                "metrics": {
                    "sku_nbr": current_sku,
                    "sku_name": sku_name,
                    "sub_class": sub_class,
                    "status": "skipped",
                    "reason": f"best model failed during final refit: {e}",
                    "n_months": n_months,
                    "best_model_type": best_model_type,
                    "best_avg_mape": best_avg_mape,
                    "best_params": best_params
                },
                "model_entry": None
            }

        future_months = get_future_months(
            temp_ts["month"].max(),
            forecast_horizon
        )

        forecast_output = pd.DataFrame({
            "sku_nbr": current_sku,
            "sub_class": sub_class,
            "sales_units": final_forecast_values,
            "month": future_months,
            "sku_name": sku_name,
            "model_type": best_model_type
        })

        forecast_output = forecast_output[
            ["sku_nbr", "sub_class", "sales_units", "month", "sku_name", "model_type"]
        ]

        metrics_output = {
            "sku_nbr": current_sku,
            "sku_name": sku_name,
            "sub_class": sub_class,
            "status": "forecasted",
            "n_months": n_months,
            "best_model_type": best_model_type,
            "best_avg_mape": best_avg_mape,
            "best_fold_mapes": best_fold_mapes,
            "n_successful_folds": best_n_successful_folds,
            "best_params": best_params,
            "n_candidate_models": len(candidate_results)
        }

        if save_model_objects:
            model_entry = {
                "model_type": best_model_type,
                "model": final_model,
                "params": best_params,
                "avg_rolling_mape": best_avg_mape,
                "fold_mapes": best_fold_mapes
            }
        else:
            model_entry = {
                "model_type": best_model_type,
                "model": None,
                "params": best_params,
                "avg_rolling_mape": best_avg_mape,
                "fold_mapes": best_fold_mapes
            }

        return {
            "sku_nbr": current_sku,
            "forecast": forecast_output,
            "metrics": metrics_output,
            "model_entry": model_entry
        }

    except Exception as e:
        return {
            "sku_nbr": current_sku,
            "forecast": None,
            "metrics": {
                "sku_nbr": current_sku,
                "sku_name": None,
                "sub_class": None,
                "status": "skipped",
                "reason": f"unexpected error: {e}",
                "n_months": None
            },
            "model_entry": None
        }

In [ ]:
def run_parallel_forecasts(
    sku_list,
    n_jobs=None,
    save_model_objects=False,
    backend="loky",
    verbose=10
):
    if n_jobs is None:
        n_jobs = max(1, os.cpu_count() - 1)

    start_time = time.time()

    results = Parallel(
        n_jobs=n_jobs,
        backend=backend,
        verbose=verbose
    )(
        delayed(process_one_sku)(
            current_sku=sku,
            save_model_objects=save_model_objects
        )
        for sku in sku_list
    )

    elapsed_seconds = time.time() - start_time

    all_forecasts = []
    all_model_metrics = []
    all_models = {}

    for result in results:

        if result["forecast"] is not None:
            all_forecasts.append(result["forecast"])

        if result["metrics"] is not None:
            all_model_metrics.append(result["metrics"])

        if result["model_entry"] is not None:
            all_models[result["sku_nbr"]] = result["model_entry"]

    if len(all_forecasts) > 0:
        forecast_results_df = pd.concat(all_forecasts, ignore_index=True)
    else:
        forecast_results_df = pd.DataFrame(
            columns=["sku_nbr", "sub_class", "sales_units", "month", "sku_name", "model_type"]
        )

    model_metrics_df = pd.DataFrame(all_model_metrics)

    runtime_summary = {
        "n_skus": len(sku_list),
        "n_jobs": n_jobs,
        "elapsed_seconds": elapsed_seconds,
        "elapsed_minutes": elapsed_seconds / 60,
        "elapsed_hours": elapsed_seconds / 3600,
        "forecast_rows_created": len(forecast_results_df),
        "skus_successfully_forecasted": (
            model_metrics_df[model_metrics_df["status"] == "forecasted"].shape[0]
            if len(model_metrics_df) > 0
            else 0
        ),
        "skus_skipped": (
            model_metrics_df[model_metrics_df["status"] == "skipped"].shape[0]
            if len(model_metrics_df) > 0
            else 0
        )
    }

    return forecast_results_df, model_metrics_df, all_models, runtime_summary

In [ ]:
# Test cell to approximate time 
test_sku_count = 30
test_sku_list = list(unique_sku_nbr)[:test_sku_count]

test_n_jobs = min(n_jobs, test_sku_count)

test_forecast_results_df, test_model_metrics_df, test_all_models, test_runtime_summary = run_parallel_forecasts(
    sku_list=test_sku_list,
    n_jobs=test_n_jobs,
    save_model_objects=False,
    backend="loky",
    verbose=10
)

test_elapsed_minutes = test_runtime_summary["elapsed_minutes"]

estimated_total_minutes = (
    test_elapsed_minutes / test_sku_count
) * max_skus

estimated_total_hours = estimated_total_minutes / 60

print(f"Test SKUs: {test_sku_count}")
print(f"Parallel jobs used: {test_n_jobs}")
print(f"Elapsed minutes for test: {test_elapsed_minutes:.2f}")
print(f"Estimated minutes for {max_skus} SKUs: {estimated_total_minutes:.2f}")
print(f"Estimated hours for {max_skus} SKUs: {estimated_total_hours:.2f}")

print(f"Test forecast rows created: {len(test_forecast_results_df)}")
print(
    "Test SKUs successfully forecasted:",
    test_model_metrics_df[test_model_metrics_df["status"] == "forecasted"].shape[0]
)
print(
    "Test SKUs skipped:",
    test_model_metrics_df[test_model_metrics_df["status"] == "skipped"].shape[0]
)

[Parallel(n_jobs=15)]: Using backend LokyBackend with 15 concurrent workers.


In [ ]:
#Full Run - ONLY DO THIS IF YOUR PC CAN HANDLE IT
sku_list = list(unique_sku_nbr)[:max_skus]

forecast_results_df, model_metrics_df, all_models, runtime_summary = run_parallel_forecasts(
    sku_list=sku_list,
    n_jobs=n_jobs,
    save_model_objects=SAVE_MODEL_OBJECTS,
    backend="loky",
    verbose=10
)

print(f"SKUs processed: {runtime_summary['n_skus']}")
print(f"Parallel jobs used: {runtime_summary['n_jobs']}")
print(f"Elapsed minutes: {runtime_summary['elapsed_minutes']:.2f}")
print(f"Forecast rows created: {runtime_summary['forecast_rows_created']}")
print(f"SKUs successfully forecasted: {runtime_summary['skus_successfully_forecasted']}")
print(f"SKUs skipped: {runtime_summary['skus_skipped']}")

In [ ]:
forecast_results_df.to_csv('data/first_166_sku_forecast_results.csv', index=False)
model_metrics_df.to_csv('data/first_166_sku_model_metrics.csv', index=False)
all_models.to_csv('data/first_166_sku_all_models.csv', index=False)
runtime_summary.to_csv('data/first_166_sku_runtime_summary.csv', index=False)